## proceso para optener la calificacion camel 
* se optienen los pesos
* se optiene la calificacion con los rangos de los cuartiles
* se hace el modelo camel
* se optiene la calificacion 

In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import pandas as pd

## PCA

In [35]:
def pesos_pca_grupo_coops(
    df,
    lista_coops,
    col_nombre="ID_cooperativa",
    n_componentes=3
):
    
    # filtrar cooperativas
    df_filtrado = df[df[col_nombre].isin(lista_coops)].copy()
    
    if df_filtrado.empty:
        raise ValueError("No hay datos para las cooperativas indicadas")
    
    # LIMPIEZA 
    df_filtrado["valor"] = (
        df_filtrado["valor"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    
    df_filtrado["valor"] = pd.to_numeric(df_filtrado["valor"], errors="coerce")
    
    # pivot (matriz indicadores)
    df_pivot = df_filtrado.pivot_table(
        index=col_nombre,
        columns="ID_indicador",
        values="valor",
        aggfunc="mean"
    )
    
    # imputar faltantes
    imputer = SimpleImputer(strategy="mean")
    X_imputed = imputer.fit_transform(df_pivot)
    
    # escalar
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_imputed)
    
    # PCA
    pca = PCA(n_components=min(n_componentes, df_pivot.shape[1]))
    pca.fit(X_scaled)
    
    loadings = pd.DataFrame(
        pca.components_.T,
        index=df_pivot.columns
    )
    
    pesos = (loadings**2).mean(axis=1)
    
    # porcentaje
    pesos = (pesos / pesos.sum()) * 100
    
    return pesos.sort_values(ascending=False)

In [38]:
datos=pd.read_excel("../tablas/registros_categorizados_con_IRL.xlsx")
datos=datos.loc[datos["ano"]>=2020]
datos.head()

,ID_registro,ID_indicador,ID_cooperativa,ano,mes,valor,categoria
0,25345,Quebranto Patrimonial,CAJA COOPERATIVA CREDICOOP,2020,1,0.0,Medianas
1,25346,Quebranto Patrimonial,CAJA COOPERATIVA PETROLERA,2020,1,0.0,Megas
2,25347,Quebranto Patrimonial,CASA NACIONAL DEL PROFESOR,2020,1,0.0,Megas
3,25348,Quebranto Patrimonial,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,2020,1,0.0,Grandes
4,25349,Quebranto Patrimonial,COOPERATIVA AHORRO Y CREDITO GOMEZ PLATA LTDA.,2020,1,0.0,Medianas


# Selecciona una categoria

In [53]:
datos["categoria"].unique()

array(['Medianas', 'Megas', 'Grandes', 'Micro 2', 'Pequeñas', 'Micro 1',
       'Top 4'], dtype=object)

In [79]:
datos_copia = datos.copy()
#datos_copia=datos_copia.loc[datos_copia["categoria"]=="Micro 1"]

In [80]:
names_coop = datos_copia["ID_cooperativa"].tolist()
pesos = pesos_pca_grupo_coops(df=datos_copia, lista_coops=names_coop)
pesos

ID_indicador
Quebranto Patrimonial                                                                     11.267600
Indicador de Margen Financiero de Operación                                               10.464778
Estructura de Balance                                                                      9.044563
Indicador de rentabilidad sobre recursos propios - ROE                                     8.924425
Indicador de Cobertura de la Cartera Total en Riesgo                                       8.074197
Relación entre el Capital Institucional y el Activo Total                                  7.535087
Indicador de rentabilidad sobre el capital invertido - ROIC                                6.587928
Indicador de margen neto                                                                   6.248995
Indicador de calidad por riesgo                                                            6.185444
Indicador de Margen Operacional                                                        

## Calificacion 

In [81]:
"""
def tabla_percentiles_globales(df: pd.DataFrame):
    if df.empty:
        return pd.DataFrame()

    df = df.copy()

    df['percentil'] = (
        df
        .groupby('ID_indicador')['valor']
        .rank(pct=True) * 100
    )

    df['decil'] = (df['percentil'] // 10).astype(int) + 1
    df['decil'] = df['decil'].clip(upper=10)  

    tabla = (
        df
        .groupby(['ID_indicador', 'decil'])['valor']
        .max()
        .unstack(fill_value=None)
    )

    tabla.columns = [f"P{int(col*10)}" for col in tabla.columns]

    return tabla
"""
import pandas as pd
import numpy as np

def tabla_percentiles_globales(df: pd.DataFrame):
    if df.empty:
        return pd.DataFrame()

    df = df.copy()

    # 🔹 Filtrar valores positivos (evita que los ceros dañen la distribución)
    df = df[df["valor"] > 0]

    if df.empty:
        return pd.DataFrame()

    # 🔹 Transformación logarítmica para datos sesgados
    df["valor_log"] = np.log1p(df["valor"])

    # 🔹 Percentiles reales
    percentiles = [i/10 for i in range(1, 11)]

    tabla = (
        df
        .groupby("ID_indicador")["valor_log"]
        .quantile(percentiles)
        .unstack()
    )

    # 🔹 Volver a escala original (opcional pero recomendado)
    tabla = np.expm1(tabla)

    # 🔹 Renombrar columnas
    tabla.columns = [f"P{int(p*100)}" for p in percentiles]

    return tabla

In [82]:
tabla_percentiles_globales(datos_copia)

,P10,P20,P30,P40,P50,P60,P70,P80,P90,P100
ID_indicador,,,,,,,,,,
Activo Productivo,7.476410e-01,7.878870e-01,8.159145e-01,8.388970e-01,8.586137e-01,8.761490e-01,8.933146e-01,9.137971e-01,9.408540e-01,1.036441e+00
Estructura de Balance,1.128537e+00,1.223552e+00,1.318343e+00,1.402124e+00,1.505142e+00,1.642533e+00,1.940395e+00,2.343995e+00,3.314739e+00,1.076551e+01
IRL,4.351221e+08,7.770754e+08,1.196518e+09,1.857635e+09,2.845897e+09,4.081749e+09,5.977910e+09,1.010317e+10,1.809137e+10,1.523502e+11
Indicador de Cobertura de la Cartera Total en Riesgo,2.624556e-02,3.131550e-02,3.691487e-02,4.300063e-02,5.120888e-02,5.970982e-02,7.105703e-02,8.168027e-02,1.033379e-01,2.357008e-01
Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,2.899093e-01,3.823889e-01,4.573583e-01,5.198650e-01,5.750658e-01,6.310358e-01,6.887638e-01,7.525455e-01,8.367417e-01,2.702713e+00
Indicador de Margen Financiero de Operación,5.485944e-01,6.263756e-01,6.742957e-01,7.156150e-01,7.482593e-01,7.800136e-01,8.214209e-01,8.641413e-01,9.109892e-01,9.979841e-01
Indicador de Margen Operacional,2.649096e-02,5.290899e-02,8.055507e-02,1.084299e-01,1.350052e-01,1.621494e-01,1.931464e-01,2.324284e-01,2.894419e-01,7.104877e-01
Indicador de calidad por riesgo,2.892688e-02,3.906569e-02,4.877062e-02,5.668800e-02,6.546496e-02,7.768414e-02,9.035570e-02,1.087275e-01,1.469025e-01,1.016518e+00
Indicador de calidad por riesgo con castigos,2.440225e-03,5.587256e-03,1.228681e-02,1.983847e-02,2.713140e-02,3.820623e-02,4.949595e-02,6.731829e-02,9.814466e-02,3.148424e-01


In [83]:
import pandas as pd
import numpy as np

def calcular_calificaciones_formato_lista(df: pd.DataFrame):
    if df.empty:
        return []

    columnas_necesarias = {
        'ID_indicador', 'ID_cooperativa', 'valor'
    }
    if not columnas_necesarias.issubset(df.columns):
        raise ValueError("Faltan columnas necesarias")

    df = df.copy()

    resultados_temp = []

    for indicador in df['ID_indicador'].unique():
        df_ind = df[df['ID_indicador'] == indicador].copy()

        # 🔹 Separar ceros (opcional pero recomendado)
        df_ind = df_ind[df_ind["valor"] > 0]

        if df_ind.empty:
            continue

        # 🔹 Transformación log
        df_ind["valor_log"] = np.log1p(df_ind["valor"])

        # 🔹 Calcular percentiles reales (umbrales)
        percentiles = [i/10 for i in range(1, 11)]
        thresholds = df_ind["valor_log"].quantile(percentiles).values

        # 🔹 Función para asignar calificación basada en thresholds
        def asignar_calificacion(valor_log):
            for i, t in enumerate(thresholds):
                if valor_log <= t:
                    return i + 1
            return 10

        df_ind["calificacion"] = df_ind["valor_log"].apply(asignar_calificacion)

        # 🔹 Promedio por cooperativa
        promedio = (
            df_ind
            .groupby('ID_cooperativa')['calificacion']
            .mean()
            .round()
            .astype(int)
        )

        for coop, calif in promedio.items():
            resultados_temp.append({
                "ID_cooperativa": str(coop),
                "ID_indicador": str(indicador),
                "calificacion": int(calif)
            })

    # 🔹 Formato final
    resultado_final = {}

    for row in resultados_temp:
        coop = row["ID_cooperativa"]
        ind = row["ID_indicador"]
        cal = row["calificacion"]

        if coop not in resultado_final:
            resultado_final[coop] = {
                "ID_cooperativa": coop,
                "calificaciones": {}
            }

        resultado_final[coop]["calificaciones"][ind] = cal

    return list(resultado_final.values())

In [84]:
calificaciones = calcular_calificaciones_formato_lista(datos_copia)
calificaciones

[{'ID_cooperativa': 'CAJA COOPERATIVA CREDICOOP',
  'calificaciones': {'Quebranto Patrimonial': 5,
   'Relación Solvencia': 5,
   'Relación entre Aportes sociales mínimos no reducibles y Capital Social': 9,
   'Relación entre el Capital Institucional y el Activo Total': 6,
   'Indicador de Cobertura de la Cartera Total en Riesgo': 6,
   'Indicador de calidad por riesgo': 8,
   'Indicador de calidad por riesgo con castigos': 10,
   'Activo Productivo': 5,
   'Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo': 6,
   'Estructura de Balance': 7,
   'Indicador de Margen Financiero de Operación': 6,
   'Indicador de Margen Operacional': 3,
   'Indicador de relación entre las obligaciones financieras y el pasivo total': 5,
   'Indicador de margen neto': 2,
   'Indicador de rentabilidad sobre el capital invertido - ROIC': 2,
   'Indicador de rentabilidad sobre recursos propios - ROE': 2,
   'IRL': 5}},
 {'ID_cooperativa': 'CAJA COOPERATIVA PETROLERA',
  'c

In [85]:
def calcular_camel(lista, pesos):
    # Normalizar pesos
    pesos_norm = pesos / pesos.sum()

    resultados = []

    for coop in lista:
        nombre = coop["ID_cooperativa"]
        califs = coop["calificaciones"]

        score = 0

        for indicador, peso in pesos_norm.items():
            calificacion = califs.get(indicador, 0)  # si falta, toma 0
            score += calificacion * peso

        resultados.append({
            "ID_cooperativa": nombre,
            "CAMEL_score": score
        })

    return pd.DataFrame(resultados)


In [86]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
camel_coop=calcular_camel(calificaciones, pesos)
camel_coop.sort_values("CAMEL_score", ascending=False)

,ID_cooperativa,CAMEL_score
79,COOPERATIVA TELEPOSTAL LTDA,7.695465
66,COOPERATIVA MULTIACTIVA DE EL PAUJIL CAQUETA L...,7.625833
7,COOPERATIVA DE AHORRO Y CREDITO BERLIN,7.491943
18,COOPERATIVA DE AHORRO Y CREDITO DE SANTANDER L...,7.337691
54,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ...,7.191536
61,COOPERATIVA LATINOAMERICANA DE AHORRO Y CREDITO,7.122393
50,COOPERATIVA DE YARUMAL,7.072095
58,COOPERATIVA INTEGRAL AGROPECUARIA LA PAZ LTDA,6.872645
86,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURS...,6.698130
10,COOPERATIVA DE AHORRO Y CREDITO COOMPARTIR,6.687476
